# Experiment 1 (Transient) — 2D Time-Dependent Heat Conduction: FNO3d Training

**Project:** AI-Powered Virtual Mechanical Engineering Laboratory
**Author:** Oluwatobi Ademola, Edo State University

Trains a **3D Fourier Neural Operator** that maps physical parameters, the boundary
configuration and the initial temperature field to the **entire space-time solution**
$T(x, y, t)$ in a single forward pass — no autoregressive rollout, no solver in the loop.

### Governing equation
$$\rho c_p \frac{\partial T}{\partial t} = \nabla\cdot(k\,\nabla T) + Q \quad\text{on } [0,1]^2,\; t\in(0,2]$$

### Inputs / output
| | |
|---|---|
| **Continuous params** | $k,\ Q,\ T_\text{left},\ T_\text{right},\ \rho c_p$ |
| **Categorical** | boundary config (5 types, one-hot) |
| **Field input** | initial condition $T(x,y,0)$ |
| **Output** | $T(x,y,t)$ on a $21\times33\times33$ ($t,y,x$) grid |

### Why FNO3d (single-shot) over autoregressive 2D
- one forward pass returns the whole trajectory → best fit for real-time web playback
- no error accumulation across the 20 time steps
- matches the snapshot dataset produced by `generate_transient.py`

### Performance — direct PyTorch tuned to match NVIDIA PhysicsNeMo
- TF32 matmuls + cuDNN autotune
- AMP (bf16/fp16) with the FFT pinned to fp32 (`torch.fft` has no half support)
- fused Adam optimiser
- GPU-resident data with on-device batching (no DataLoader transfer bottleneck)
- `torch.compile` graph fusion (with an automatic eager fallback)

## 1. Setup & performance flags

In [ ]:
import os, time, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Performance flags (match PhysicsNeMo defaults) ----
torch.backends.cuda.matmul.allow_tf32 = True   # TF32 matmuls on Ampere+
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True          # autotune conv kernels (shapes are fixed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')

torch.manual_seed(42)
np.random.seed(42)

## 2. Mount Google Drive & load dataset

In [ ]:
import os
from google.colab import drive

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# ── Edit these to match your Drive layout ──
DATA_DIR  = '/content/drive/MyDrive/vme-lab/data/processed/heat_conduction_transient'
DRIVE_OUT = '/content/drive/MyDrive/vme-lab/models/heat_conduction'
# ───────────────────────────────────────────

In [ ]:
# Orderings — must match generate_transient.py exactly
PARAM_NAMES = ['k', 'Q', 'T_left', 'T_right', 'rho_cp']
IC_TYPES    = ['uniform', 'gaussian_hotspot', 'linear_ramp']
BC_CONFIGS  = ['dirichlet_lr', 'dirichlet_all', 'neumann_top_flux',
               'robin_right', 'insulated_sides_source']

def load_split(split):
    z = np.load(os.path.join(DATA_DIR, f'{split}.npz'), allow_pickle=True)
    return (z['params'].astype(np.float32),   # (N, 5)
            z['fields'].astype(np.float32),   # (N, 21, 33, 33)
            z['t_grid'].astype(np.float32),   # (21,)
            z['ic_types'],                    # (N,)  str
            z['bc_configs'])                  # (N,)  str

params_tr, fields_tr, t_grid, ic_tr, bc_tr = load_split('train')
params_va, fields_va, _,      ic_va, bc_va = load_split('val')
params_te, fields_te, _,      ic_te, bc_te = load_split('test')

T, H, W = fields_tr.shape[1:]    # 21, 33, 33  (time, y, x)
print(f'train {fields_tr.shape}   val {fields_va.shape}   test {fields_te.shape}')
print(f'space-time grid: T={T}  H={H}  W={W}')

## 3. Normalisation & encodings

In [ ]:
# Continuous params → [0, 1] using TRAIN stats only (no leakage)
param_min = params_tr.min(axis=0)
param_max = params_tr.max(axis=0)
def norm_params(p):
    return (p - param_min) / (param_max - param_min + 1e-8)

# Fields → zero-mean unit-std using TRAIN stats (global scalars)
field_mean = float(fields_tr.mean())
field_std  = float(fields_tr.std())
def norm_fields(f):
    return (f - field_mean) / (field_std + 1e-8)
def denorm_fields(f):
    return f * (field_std + 1e-8) + field_mean

# Boundary config → one-hot (len = 5)
bc_index = {name: i for i, name in enumerate(BC_CONFIGS)}
def bc_onehot(bc_arr):
    idx = np.array([bc_index[str(b)] for b in bc_arr], dtype=np.int64)
    oh = np.zeros((len(idx), len(BC_CONFIGS)), dtype=np.float32)
    oh[np.arange(len(idx)), idx] = 1.0
    return oh

print(f'param_min  {param_min}')
print(f'param_max  {param_max}')
print(f'field_mean {field_mean:.2f} K   field_std {field_std:.2f} K')

# Persist constants for the FastAPI inference wrapper
np.savez('norm_stats_transient.npz',
         param_min=param_min, param_max=param_max,
         field_mean=np.array([field_mean]), field_std=np.array([field_std]),
         param_names=np.array(PARAM_NAMES), bc_configs=np.array(BC_CONFIGS),
         ic_types=np.array(IC_TYPES), t_grid=t_grid)
print('saved norm_stats_transient.npz')

## 4. GPU-resident tensors & on-device batching

The whole dataset (~30 MB of fields) lives on the GPU. Each batch's 14-channel input is
assembled on-device from compact tensors via broadcasting, so there is **no host→device
copy in the training loop** — the main throughput win for small models like this one.

**Input channels (14):** 5 params · 5 BC one-hot · 1 initial-condition field · 3 coords (x, y, t).

In [ ]:
# Coordinate channels x, y, t in [0,1] over the (T,H,W) grid — shared by all samples.
# fields axis layout is (t, y, x): axis -2 = y (H), axis -1 = x (W).
xs = np.linspace(0, 1, W, dtype=np.float32)
ys = np.linspace(0, 1, H, dtype=np.float32)
ts = (t_grid / (t_grid.max() + 1e-8)).astype(np.float32)
gt, gy, gx = np.meshgrid(ts, ys, xs, indexing='ij')             # each (T,H,W)
COORD = torch.from_numpy(np.stack([gx, gy, gt], 0)).to(DEVICE)  # (3,T,H,W)

def pack(params, fields, bc_arr):
    P  = torch.from_numpy(norm_params(params)).to(DEVICE)   # (N,5)
    BC = torch.from_numpy(bc_onehot(bc_arr)).to(DEVICE)     # (N,5)
    fn = norm_fields(fields)                                # (N,T,H,W)
    IC = torch.from_numpy(fn[:, 0:1]).to(DEVICE)            # (N,1,H,W)  -> T(x,y,0)
    Y  = torch.from_numpy(fn[:, None]).to(DEVICE)           # (N,1,T,H,W)
    return P, BC, IC, Y

P_tr, BC_tr, IC_tr, Y_tr = pack(params_tr, fields_tr, bc_tr)
P_va, BC_va, IC_va, Y_va = pack(params_va, fields_va, bc_va)
P_te, BC_te, IC_te, Y_te = pack(params_te, fields_te, bc_te)

IN_CH = 5 + len(BC_CONFIGS) + 1 + 3   # = 14

def make_batch(P, BC, IC, Y, idx):
    b = idx.numel()
    p  = P[idx][:, :, None, None, None].expand(b, 5, T, H, W)
    bc = BC[idx][:, :, None, None, None].expand(b, len(BC_CONFIGS), T, H, W)
    ic = IC[idx][:, :, None, :, :].expand(b, 1, T, H, W)
    co = COORD[None].expand(b, 3, T, H, W)
    x  = torch.cat([p, bc, ic, co], dim=1).contiguous()   # (b,14,T,H,W)
    return x, Y[idx]

xb, yb = make_batch(P_tr, BC_tr, IC_tr, Y_tr, torch.arange(4, device=DEVICE))
print(f'input batch {tuple(xb.shape)}   target batch {tuple(yb.shape)}   IN_CH={IN_CH}')

## 5. FNO3d architecture

Each Fourier layer:
$$v_{\ell+1} = \sigma\!\Big(W v_\ell + \mathcal{F}^{-1}\big[R_\phi\cdot\mathcal{F}[v_\ell]\big]\Big)$$
with the spectral multiply truncated to the lowest `modes` frequencies in $(t, y, x)$.
The contraction matches PhysicsNeMo's spectral layer; the additions here are the
AMP-safe fp32 FFT and a trace-friendly forward path for export.

In [ ]:
# ---------------------------------------------------------------------------
# SpectralConv3d — rFFT over (t,y,x), keep low modes, learnable complex weights.
# 3D rFFT halves the last axis and keeps +/- frequencies on the first two, so we
# learn 4 corner blocks. FFT is forced to fp32 under AMP (no half support).
# ---------------------------------------------------------------------------
class SpectralConv3d(nn.Module):
    def __init__(self, in_ch, out_ch, m1, m2, m3):
        super().__init__()
        self.in_ch, self.out_ch = in_ch, out_ch
        self.m1, self.m2, self.m3 = m1, m2, m3
        scale = 1.0 / (in_ch * out_ch)
        def w():
            return nn.Parameter(scale * torch.rand(in_ch, out_ch, m1, m2, m3, dtype=torch.cfloat))
        self.w1, self.w2, self.w3, self.w4 = w(), w(), w(), w()

    @staticmethod
    def _mul(a, w):
        # (B,in,d,e,f) x (in,out,d,e,f) -> (B,out,d,e,f)
        return torch.einsum('bidef,iodef->bodef', a, w)

    def _spectral(self, x):
        B = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=(-3, -2, -1))
        out = torch.zeros(B, self.out_ch, x.shape[-3], x.shape[-2], x.shape[-1] // 2 + 1,
                          dtype=torch.cfloat, device=x.device)
        m1, m2, m3 = self.m1, self.m2, self.m3
        out[:, :,  :m1,  :m2, :m3] = self._mul(x_ft[:, :,  :m1,  :m2, :m3], self.w1)
        out[:, :, -m1:,  :m2, :m3] = self._mul(x_ft[:, :, -m1:,  :m2, :m3], self.w2)
        out[:, :,  :m1, -m2:, :m3] = self._mul(x_ft[:, :,  :m1, -m2:, :m3], self.w3)
        out[:, :, -m1:, -m2:, :m3] = self._mul(x_ft[:, :, -m1:, -m2:, :m3], self.w4)
        return torch.fft.irfftn(out, s=(x.shape[-3], x.shape[-2], x.shape[-1]), dim=(-3, -2, -1))

    def forward(self, x):
        # Pin FFT to fp32 only when AMP autocast is active; trace/eval takes the plain path.
        if x.is_cuda and torch.is_autocast_enabled():
            with torch.autocast('cuda', enabled=False):
                return self._spectral(x.float())
        return self._spectral(x)


class FNOBlock3d(nn.Module):
    def __init__(self, width, m1, m2, m3):
        super().__init__()
        self.spectral = SpectralConv3d(width, width, m1, m2, m3)
        self.skip = nn.Conv3d(width, width, 1)

    def forward(self, x):
        return F.gelu(self.spectral(x) + self.skip(x))


class FNO3d(nn.Module):
    def __init__(self, in_ch=14, out_ch=1, width=32, modes=(6, 10, 10), n_layers=4):
        super().__init__()
        m1, m2, m3 = modes
        self.lift = nn.Conv3d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNOBlock3d(width, m1, m2, m3) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Conv3d(width, 128, 1), nn.GELU(), nn.Conv3d(128, out_ch, 1))

    def forward(self, x):
        x = self.lift(x)
        for blk in self.blocks:
            x = blk(x)
        return self.proj(x)


WIDTH, MODES, N_LAYERS = 32, (6, 10, 10), 4
model = FNO3d(in_ch=IN_CH, width=WIDTH, modes=MODES, n_layers=N_LAYERS).to(DEVICE)
n_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'FNO3d  width={WIDTH}  modes={MODES}  layers={N_LAYERS}  params={n_param:,}')
with torch.no_grad():
    d = torch.zeros(2, IN_CH, T, H, W, device=DEVICE)
    print('forward sanity:', tuple(d.shape), '->', tuple(model(d).shape))

## 6. Loss, optimiser & AMP

In [ ]:
def relative_l2_loss(pred, target):
    # scale-invariant L2 over the full space-time field, averaged over the batch.
    # computed in fp32 for numerical stability under AMP.
    pred = pred.float(); target = target.float()
    diff = (pred - target).reshape(pred.shape[0], -1)
    tgt  = target.reshape(target.shape[0], -1)
    return (diff.norm(dim=1) / (tgt.norm(dim=1) + 1e-8)).mean()

EPOCHS, LR, WEIGHT_DECAY, BATCH = 300, 1e-3, 1e-4, 8
USE_COMPILE = True

fused_ok = DEVICE.type == 'cuda'
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=fused_ok)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp    = DEVICE.type == 'cuda'
amp_dtype  = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = use_amp and (amp_dtype == torch.float16)
scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)
print(f'AMP={use_amp}  dtype={amp_dtype}  grad_scaler={use_scaler}')

model_c = torch.compile(model) if (USE_COMPILE and DEVICE.type == 'cuda') else model
print(f'torch.compile requested={USE_COMPILE and DEVICE.type == "cuda"}')

## 7. Training loop

In [ ]:
n_train = P_tr.shape[0]
steps   = math.ceil(n_train / BATCH)
history = {'train': [], 'val': []}
best_val = float('inf')
CKPT = 'transient_fno_best.pt'

# Compile is lazy — the first call may raise if a graph break is unsupported.
# Warm up once and fall back to eager so the run never dies on a compile issue.
try:
    _x, _y = make_batch(P_tr, BC_tr, IC_tr, Y_tr, torch.arange(min(BATCH, n_train), device=DEVICE))
    with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
        _ = model_c(_x)
    print('warmup ok')
except Exception as e:
    print(f'compiled model failed ({type(e).__name__}: {e}); using eager model')
    model_c = model

@torch.no_grad()
def eval_loss(P, BC, IC, Y):
    model.eval()
    n = P.shape[0]; tot = 0.0; nb = 0
    for s in range(0, n, BATCH):
        idx = torch.arange(s, min(s + BATCH, n), device=DEVICE)
        xb, yb = make_batch(P, BC, IC, Y, idx)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
            pred = model_c(xb)
        tot += relative_l2_loss(pred, yb).item(); nb += 1
    return tot / nb

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    perm = torch.randperm(n_train, device=DEVICE)
    run = 0.0
    for s in range(steps):
        idx = perm[s * BATCH:(s + 1) * BATCH]
        xb, yb = make_batch(P_tr, BC_tr, IC_tr, Y_tr, idx)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
            loss = relative_l2_loss(model_c(xb), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        run += loss.item()
    scheduler.step()
    tr = run / steps
    va = eval_loss(P_va, BC_va, IC_va, Y_va)
    history['train'].append(tr); history['val'].append(va)
    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), CKPT)
    if epoch == 1 or epoch % 10 == 0:
        print(f'epoch {epoch:>3}/{EPOCHS}  train={tr:.4f}  val={va:.4f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  {time.time()-t0:.1f}s')

print(f'\nbest val rel-L2 = {best_val:.4f}  -> {CKPT}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(history['train'], label='train')
ax.semilogy(history['val'], label='val')
ax.set_xlabel('epoch'); ax.set_ylabel('relative L2'); ax.legend()
ax.set_title('FNO3d training — transient heat conduction')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.savefig('training_curve_transient.png', dpi=150); plt.show()
print('saved training_curve_transient.png')

## 8. Test-set evaluation

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

@torch.no_grad()
def predict(P, BC, IC, Y):
    n = P.shape[0]; outs = []
    for s in range(0, n, BATCH):
        idx = torch.arange(s, min(s + BATCH, n), device=DEVICE)
        xb, _ = make_batch(P, BC, IC, Y, idx)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
            outs.append(model(xb).float().cpu())
    return torch.cat(outs)   # (n,1,T,H,W) normalised

pred_te = predict(P_te, BC_te, IC_te, Y_te)
targ_te = Y_te.cpu()

rel    = relative_l2_loss(pred_te, targ_te).item()
pred_K = denorm_fields(pred_te.numpy()[:, 0])   # (n,T,H,W) Kelvin
targ_K = denorm_fields(targ_te.numpy()[:, 0])
rmse_K = float(np.sqrt(((pred_K - targ_K) ** 2).mean()))
maxe_K = float(np.abs(pred_K - targ_K).max())
print(f'test rel-L2 = {rel:.4f}')
print(f'test RMSE   = {rmse_K:.3f} K')
print(f'test maxerr = {maxe_K:.3f} K')

# per-timestep relative L2 — shows error growth across the rollout
num = np.sqrt(((pred_K - targ_K) ** 2).reshape(pred_K.shape[0], T, -1).sum(-1))
den = np.sqrt((targ_K ** 2).reshape(targ_K.shape[0], T, -1).sum(-1)) + 1e-8
per_t = (num / den).mean(0)   # (T,)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_grid, per_t, marker='o')
ax.set_xlabel('time t (s)'); ax.set_ylabel('mean relative L2')
ax.set_title('Prediction error vs simulation time (test set)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('error_vs_time.png', dpi=150); plt.show()
print('saved error_vs_time.png')

In [ ]:
# Two test samples: ground truth (top) vs FNO (bottom) at t = 0, T/2, T_end
sel  = [0, 1]
cols = [0, T // 2, T - 1]
fig, axes = plt.subplots(len(sel) * 2, len(cols), figsize=(4 * len(cols), 4 * len(sel)))
for r, i in enumerate(sel):
    vmin = min(targ_K[i].min(), pred_K[i].min())
    vmax = max(targ_K[i].max(), pred_K[i].max())
    for c, step in enumerate(cols):
        ax_gt, ax_pr = axes[2 * r, c], axes[2 * r + 1, c]
        im = ax_gt.imshow(targ_K[i, step], origin='lower', cmap='hot', vmin=vmin, vmax=vmax)
        ax_pr.imshow(pred_K[i, step], origin='lower', cmap='hot', vmin=vmin, vmax=vmax)
        ax_gt.set_title(f'sample {i}  GT   t={t_grid[step]:.1f}s', fontsize=9)
        ax_pr.set_title(f'sample {i}  FNO  t={t_grid[step]:.1f}s', fontsize=9)
        plt.colorbar(im, ax=ax_gt, shrink=0.7, label='K')
plt.savefig('test_predictions_transient.png', dpi=150, bbox_inches='tight'); plt.show()
print('saved test_predictions_transient.png')

## 9. Inference helper — run from parameters alone

For deployment the surrogate must run from `(params, ic_type, bc_config)` with **no solver**.
The initial-condition field is synthesised here with the exact formulas from
`generate_transient.py`, then fed to the model. `predict_trajectory` returns the full
$T(x,y,t)$ field the FastAPI backend will serve.

In [ ]:
# IC fields on the (H,W) grid — Xg = x (along W), Yg = y (along H). Mirrors generate_transient.py.
Xg, Yg = np.meshgrid(np.linspace(0, 1, W, dtype=np.float32),
                     np.linspace(0, 1, H, dtype=np.float32))

def build_ic_field(ic_type, T_left, T_right):
    if ic_type == 'uniform':
        return np.full((H, W), T_right, dtype=np.float32)
    if ic_type == 'gaussian_hotspot':
        T_bg = (T_left + T_right) / 2.0
        A, sigma = 0.3 * T_bg, 0.1
        return (T_bg + A * np.exp(-((Xg - 0.5) ** 2 + (Yg - 0.5) ** 2) / (2 * sigma ** 2))).astype(np.float32)
    if ic_type == 'linear_ramp':
        return (T_left + (T_right - T_left) * Xg).astype(np.float32)
    raise ValueError(f'unknown ic_type: {ic_type}')

@torch.no_grad()
def predict_trajectory(k, Q, T_left, T_right, rho_cp, ic_type, bc_config):
    model.eval()
    p  = torch.from_numpy(norm_params(np.array([[k, Q, T_left, T_right, rho_cp]], np.float32))).to(DEVICE)
    bc = torch.from_numpy(bc_onehot(np.array([bc_config]))).to(DEVICE)
    ic = torch.from_numpy(norm_fields(build_ic_field(ic_type, T_left, T_right))[None, None]).to(DEVICE)
    dummyY = torch.zeros(1, 1, T, H, W, device=DEVICE)
    xb, _ = make_batch(p, bc, ic, dummyY, torch.arange(1, device=DEVICE))
    with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
        out = model(xb).float().cpu().numpy()[0, 0]   # (T,H,W) normalised
    return denorm_fields(out)

traj = predict_trajectory(k=10.0, Q=500.0, T_left=600.0, T_right=300.0,
                          rho_cp=1_000_000.0, ic_type='gaussian_hotspot',
                          bc_config='dirichlet_lr')
print('trajectory shape:', traj.shape, f'  T range [{traj.min():.1f}, {traj.max():.1f}] K')

## 10. Export — TorchScript (.pt) and ONNX

In [ ]:
# TorchScript is the primary artefact: tracing handles torch.fft reliably.
model.eval()
example = torch.zeros(1, IN_CH, T, H, W, device=DEVICE)
with torch.no_grad():
    traced = torch.jit.trace(model, example)
traced.save('transient_fno.pt')
print('saved transient_fno.pt (TorchScript)')

rt = torch.jit.load('transient_fno.pt', map_location=DEVICE)
with torch.no_grad():
    diff = (model(example) - rt(example)).abs().max().item()
print(f'TorchScript round-trip max diff: {diff:.2e}')

In [ ]:
# ONNX is best-effort: 3D rFFT/irFFT are not reliably supported by the exporter.
ONNX_PATH = 'transient_fno.onnx'
try:
    torch.onnx.export(model, example, ONNX_PATH, opset_version=17,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                      do_constant_folding=True)
    import onnxruntime as ort
    sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    o = sess.run(None, {'input': example.cpu().numpy()})[0]
    print('ONNX max diff:', np.abs(o - model(example).detach().cpu().numpy()).max())
except Exception as e:
    print(f'ONNX export failed ({type(e).__name__}: {e}).')
    print('Use the TorchScript .pt for serving (torch.jit.load) — FFT ops do not export to ONNX cleanly.')

## 11. Copy artefacts to Google Drive

In [ ]:
import shutil
os.makedirs(DRIVE_OUT, exist_ok=True)
arts = ['transient_fno_best.pt', 'transient_fno.pt', 'norm_stats_transient.npz',
        'training_curve_transient.png', 'error_vs_time.png', 'test_predictions_transient.png']
if os.path.exists('transient_fno.onnx'):
    arts.append('transient_fno.onnx')
for f in arts:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(DRIVE_OUT, f)); print('copied', f)
    else:
        print('skip (missing)', f)
print('done')